# CSE2530 Computational Intelligence
## Assignment 1: Ant Colony Optimization and Genetic Algorithms

<div style="background-color:#f1be3e">

_Fill in your group number **from Brightspace**, names, and student numbers._
    
|    Group   |           X          |
|------------|----------------------|
| Luca Bixade |        6129080      |
| Student B  |        XXXXXXX       |
| Student C  |        XXXXXXX       |
| Student D  |        XXXXXXX       |

#### Imports

In [1]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""
import numpy as np
import random
import sys
import time

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Coordinate import Coordinate
from Direction import Direction
from PathSpecification import PathSpecification
from Route import Route
from SurroundingPheromone import SurroundingPheromone
from TSPData import TSPData

## Part 1: The Travelling Robot Problem
### 1.1 Problem Analysis
#### Question 1:

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 2

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 3

<div style="background-color:#f1be3e">

_Write your answer here._

### 1.2 Genetic Algorithm

In [8]:
# TSP problem solver using genetic algorithms.
class GeneticAlgorithm:

    """
    Constructs a new 'genetic algorithm' object.
    @param generations: the amount of generations.
    @param pop_size: the population size.
    """
    def __init__(self, generations, pop_size):
        self.generations = generations
        self.pop_size = pop_size
    """
    This method should solve the TSP.
    @param tsp_data: the data describing the problem.
    @return the optimized product sequence.
    """
    def solve_tsp(self, tsp_data):
        pass

#### Question 4

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 5

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 6

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 7

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 8

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 9

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 10

In [0]:
# Please keep your parameters for the Genetic Algorithm easily changeable here
population_size = 20
generations = 20
persist_file = "./../data/optimal_tsp"

# Setup optimization
tsp_data = TSPData.read_from_file(persist_file)
ga = GeneticAlgorithm(generations, population_size)

# Run optimzation and write to file
solution = ga.solve_tsp(tsp_data)
tsp_data.write_action_file(solution, "./../data/tsp_solution.txt")

<div style="background-color:#f1be3e">

_Put your code extra blocks above (if any) and write your answer here._

## Part 2: Path Finding Through Ant Colony Optimization
### 2.2 Observing the Problem

#### Question 11

_What is the purpose of Ant Colony Optimization? In what settings is it typically used?_

The Ant Colony Optimization is an algorithm which helps find the shortest path covering a subset of nodes in a graph.
It is used on problems where exact solutions would require a number of computational steps that grows exponentially with the input, such as the Traveling Salesman Problem, vehicle routing, or scheduling. It is a stochastic algorithm, which is why its results are very close to the optimal solution.

#### Question 12

_Make a list of the “topographical” features you can expect in a maze that increase the
difficulty of finding the finish line and require creative solutions. Discuss at least 2._

- The high number of possible choices to be taken at a certain step
- The number of dead ends
- The possibility that, after following a certain path, we end up in a previously visited location
- Paths that seem to be getting closer to the exit but eventually lead to a dead end.

1. The high number of possible choices will affect the ease with which we find the maze exit, as the mazes we will be working with are also relatively big. One intersection with 4 branches followed immediately by another will increase the search space quickly, therefore we will need to increase the number of artificial ants based on the size and complexity of our maze.
2. The possibility of a loop is also an obstacle in finding the way out. We will need to keep track of previously visited locations for each ant and therefore to reduce the probability of revisiting them, even though it may be computationally expensive, otherwise we risk wasting considerable amounts computational effort.

#### Question 13

_Give an equation for the amount of pheromone dropped by the ants. Explain why ants need to drop the pheromones in the maze._

The equation for the pheromone update in the maze is:

$\tau_{ij} = (1 - \rho)\tau_{ij} + \sum_{k=1}^m \Delta \tau_{ij}^k$

Where $\rho$ is the evaporation constant, and $\tau_{ij}^k$ is the amount of pheromone left on edge from i to j by the ant k. The ants need to drop pheromone in the maze such that, on the next iteration, subsequent ants will know which edge to follow based on how much pheromone there is on each path. This way, good paths will attract more ants and therefore will get reinforced over time, getting closer to an optimal solution.

#### Question 14

<div style="background-color:#f1be3e">

_Write your answer here. You can also use LaTeX notation in a Jupyter Notebook._

### 2.3 Implementing the Ant Algorithm

In [44]:
# Class that represents the basic Ant functionality
class StandardAnt:

    """
    Constructor of a StandardAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification, step_limit):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random
        self.steps = 0
        self.step_limit = step_limit

    """
    Method that checks if the given position is valid (so not a wall or out of the maze).
    @param pos: position to be checked.
    """
    def is_valid_tile(self, pos):
        return self.maze.in_bounds(pos) and self.maze.walls[pos.get_x()][pos.get_y()] == 1

    """
    Method that, based on the current position, checks all surrounding directions and returns only the valid ones.
    So no walls or out of bounds tiles.
    """
    def get_valid_directions(self):

        pos = self.current_position

        directions = []

        if self.is_valid_tile(pos.add_direction(Direction.north)):
            directions.append(Direction.north)
        if self.is_valid_tile(pos.add_direction(Direction.south)):
            directions.append(Direction.south)
        if self.is_valid_tile(pos.add_direction(Direction.east)):
            directions.append(Direction.east)
        if self.is_valid_tile(pos.add_direction(Direction.west)):
            directions.append(Direction.west)

        return directions

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)

        while self.current_position != self.end and self.steps < self.step_limit:
            #look at surrounding tiles
            sum_pheromone = 0
            surrounding_pheromones = []
            directions = self.get_valid_directions()
            pos = self.current_position

            for direction in directions:
                new_pos = pos.add_direction(direction)
                surrounding_pheromones.append(self.maze.pheromones[new_pos.get_x(), new_pos.get_y()])
                sum_pheromone += self.maze.pheromones[new_pos.get_x(), new_pos.get_y()]

            direction = self.rand.choices(directions, weights=surrounding_pheromones)[0]

            self.current_position = self.current_position.add_direction(direction)
            route.add(direction)
            self.steps += 1

        terminated = self.current_position == self.end
        return route, terminated

In [24]:
# Class that holds all the maze data.
# This includes the pheromones, the open and blocked tiles in the system,
# and the starting and end coordinates for the ants.
class Maze:

    """
    Constructor of a Maze
    @param walls: array of ints representing the accessible (1) and inaccessible (0) tiles
    @param width: the width (horizontal dimension) of the Maze
    @param length: the length (vertical dimension) of the Maze
    """
    def __init__(self, walls, width, length):
        self.walls = walls
        self.length = length
        self.width = width
        self.start = None
        self.end = None
        self.pheromones = np.zeros((width, length))
        self.initialize_pheromones()

    """
    Initialize pheromones on all tiles of the Maze
    """
    def initialize_pheromones(self, init_value = 10):
        self.pheromones[self.walls == 1] = init_value



    """
    Reset the Maze for a new shortest path problem
    """
    def reset(self):
        self.initialize_pheromones()

    """
    Transform the Route param to a list of coordinates corresponding to the same route.
    @param directions_route: Route object to transform
    """
    def create_coord_route(self, directions_route):
        (x, y) = (directions_route.get_start().get_x(), directions_route.get_start().get_y())
        coordinate_route = [(x, y)]
        route = directions_route.get_route()
        for d in route:
            if d == Direction.east:
                x += 1
            elif d == Direction.north:
                y += 1
            elif d == Direction.west:
                x -= 1
            elif d == Direction.south:
                y -= 1
            coordinate_route.append((x, y))

        return coordinate_route

    """
    Update the pheromones along a certain route according to a certain Q
    @param route: the route taken by an ant
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_route(self, route, q):
        coordinate_route = self.create_coord_route(route)
        l = len(coordinate_route)
        for (x, y) in coordinate_route:
            self.pheromones[x, y] += q/l


    """
    Update pheromones for a list of routes
    @param routes: a list of routes taken by the ants
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_routes(self, routes, q):
        for r in routes:
            self.add_pheromone_route(r, q)

    """
    Evaporate pheromone
    @param rho: the evaporation factor
    """
    def evaporate(self, rho):
        self.pheromones *= (1 - rho)


    """
    Getter for the width of the maze
    @return the width of the maze
    """
    def get_width(self):
        return self.width

    """
    Getter for the length of the maze
    @return the length of the maze
    """
    def get_length(self):
        return self.length

    """
    Getter for the pheromones on a specific coordinate.
    If the position is not in bounds returns 0
    @param pos: coordinate for the poition of interest
    @return the amount of pheromone at the specified poition
    """
    def get_pheromone(self, pos):
        if self.in_bounds(pos):
            return self.pheromones[pos.get_x(), pos.get_y()]
        else:
            return 0

    """
    Returns a the amount of pheromones on the neighbouring positions (N/S/E/W)
    @param position: the coordinate where we need to check the surrounding pheromones
    @return the pheromones on the neighbouring coordinates.
    """
    def get_surrounding_pheromone(self, position):
        (x, y) = (position.get_x(), position.get_y())
        n = self.get_pheromone(Coordinate(x, y+1))
        s = self.get_pheromone(Coordinate(x, y-1))
        e = self.get_pheromone(Coordinate(x+1, y))
        w = self.get_pheromone(Coordinate(x-1, y))
        return n, e, s, w

    """
    Check whether a coordinate lies in the bounds of the current maze
    @param position: the position that we need to check
    @return true if the coordinate lies within the current maze
    """
    def in_bounds(self, position):
        return position.x_between(0, self.width) and position.y_between(0, self.length)

    """
    Representation of Maze as defined by the input file format.
    @return the human-readable representation of a maze
    """
    def __str__(self):
        string = ""
        string += str(self.width)
        string += " "
        string += str(self.length)
        string += " \n"
        for y in range(self.length):
            for x in range(self.width):
                string += str(self.walls[x][y])
                string += " "
            string += "\n"
        return string

    """
    Method that builds a maze from a file
    @param file_path: path to the file which stores the maze
    @return a maze object with pheromones initialized to 0s on inaccessible and 1s on accessible tiles
    """
    @staticmethod
    def create_maze(file_path):
        try:
            f = open(file_path, "r")
            lines = f.read().splitlines()
            dimensions = lines[0].split(" ")
            width = int(dimensions[0])
            length = int(dimensions[1])
            
            #make the maze_layout
            maze_layout = []
            for x in range(width):
                maze_layout.append([])
            
            for y in range(length):
                line = lines[y+1].split(" ")
                for x in range(width):
                    if line[x] != "":
                        state = int(line[x])
                        maze_layout[x].append(state)
            print("Ready reading maze file " + file_path)
            maze_layout = np.array(maze_layout)
            return Maze(maze_layout, width, length)
        except FileNotFoundError:
            print("Error reading maze file " + file_path)

In [25]:
# Class representing the complete ACO algorithm.
# Finds shortest path between two points in a maze according to a path specification.
class AntColonyOptimization:

    """
    Constructs a new optimization object using the ant algorithm
    @param maze: the maze (environment) for ants
    @param ants_per_gen: the number of ants per generation (between update of pheromones)
    @param generations: the total number of generations of ants (pheromone updates)
    @param q: the normalization factor for the amount of dropped pheromone
    @param evaporation: the evaporation factor for the pheromones
    """
    def __init__(self, maze, ants_per_gen, generations, q, evaporation, step_limit, threshold):
        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation
        self.step_limit = step_limit
        self.threshold = threshold

    """
    Loop that starts the shortest path process
    @param path_specification: description of the route we wish to optimize
    @return the optimized route according to the ACO algorithm
    """
    def find_shortest_route(self, path_specification):
        self.maze.reset()

        best = None
        best_size = float('inf')

        all_routes = []

        for generation in range(self.generations):

            curr_routes = []

            for ant_nr in range(self.ants_per_gen):
                ant = StandardAnt(self.maze, path_specification, step_limit = self.step_limit)

                route, terminated = ant.find_route()

                if terminated:
                    curr_routes.append(route)
                    all_routes.append(route)
                    if route.size() < best_size:

                        best = route
                        best_size = route.size()

            self.maze.add_pheromone_routes(curr_routes, self.q)
            self.maze.evaporate(self.evaporation)

        return best, len(all_routes)

In [26]:
# Please keep your parameters for the ACO easily changeable here
gen = 15
no_gen = 5
q = 1600
evap = 0.1
step_scale_factor = 10

# Construct the optimization objects
maze = Maze.create_maze("./../data/medium_maze.txt")
spec = PathSpecification.read_coordinates("./../data/medium_coordinates.txt")

max_steps = maze.width * maze.length * step_scale_factor

aco = AntColonyOptimization(maze, gen, no_gen, q, evap, max_steps, 100)

start_time = int(round(time.time() * 1000))
shortest_route, routes_found = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
print("Route size: " + str(shortest_route.size()))
print("Number of ants that got out: " + str(routes_found))

shortest_route.write_to_file("./../data/medium_solution.txt")

Ready reading maze file ./../data/medium_maze.txt


NameError: name 'StandardAnt' is not defined

### 2.4 Upgrading Your Ants with Intelligence

### Changes from the Standard Ant Model

#### 1. Loop removal

   - The ants have a memory, in which previously visited locations are stored. If such a location is encountered again by an ant, every other location that was in between these 2 moments in time will be deleted from the route and pheromone will only be deposited on the distinct tiles.
   - Empirically, the differences in results are enormous, as before we could have encountered route sizes of about 1 million steps for the hard maze, whereas now we encounter routes of at most 1200 tiles.

#### 2. Heuristic Improvement

   - To preserve explorability, we have added a heuristic which keeps track of the number of times a location was seen before. This is done so that the ants prefer new routes, and has proved to be of great help in improving the runtime of the algorithm.
   - The rule is $$weight_{(x, y)} = ph_{(x, y)}^\alpha * h^\beta$$, where $ph_{(x, y)}$ is the amount of pheromone on the tile at coordinate $(x, y)$, and $h = 1/(1+t_{(x, y)})$ is the heuristic used, in which $t_{(x, y)}$ denotes the number of times the tile at $(x, y)$ has been visited before.

#### Question 15

In [88]:
# Class that represents the intelligent Ant
class IntelligentAnt:

    """
    Constructor of an IntelligentAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification, step_limit, pheromone_exp, heuristic_exp):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random
        self.step_limit = step_limit
        self.steps = 0
        self.pheromone_exp = pheromone_exp
        self.heuristic_exp = heuristic_exp
        self.visited = [self.start]
        self.visited_set = {self.start}
        self.visited_counts = {}

    """
    Method that checks if the given position is valid (so not a wall or out of the maze).
    @param pos: position to be checked.
    """
    def is_valid_tile(self, pos):
        return self.maze.in_bounds(pos) and self.maze.walls[pos.get_x(), pos.get_y()] == 1

    def get_valid_directions(self):

        pos = self.current_position

        directions = []

        if self.is_valid_tile(pos.add_direction(Direction.north)):
            directions.append(Direction.north)
        if self.is_valid_tile(pos.add_direction(Direction.south)):
            directions.append(Direction.south)
        if self.is_valid_tile(pos.add_direction(Direction.east)):
            directions.append(Direction.east)
        if self.is_valid_tile(pos.add_direction(Direction.west)):
            directions.append(Direction.west)

        return directions

    def make_move(self, direction, route):

        next_pos = self.current_position.add_direction(direction)

        if next_pos in self.visited_set:
            index = self.visited.index(next_pos)
            for pos in self.visited[index+1:]:
                self.visited_set.remove(pos)
            self.visited = self.visited[:(index+1)]
            route.route = route.route[:index]
            self.current_position = self.visited[-1]
        else:
            self.current_position = next_pos
            self.visited.append(next_pos)
            self.visited_set.add(next_pos)
            route.add(direction)
            self.steps += 1

        return route

    def pick_direction(self):
        pos = self.current_position
        directions = self.get_valid_directions() # get all possible directions to go to
        d_weights = []
        for direction in directions:
            next_pos = pos.add_direction(direction) # for each new possible position
            pheromone_factor = np.pow(self.maze.pheromones[next_pos.get_x(), next_pos.get_y()], self.pheromone_exp)
            heuristic_factor = np.pow(1/(1 + self.visited_counts.get(next_pos, 0)), self.heuristic_exp)
            d_weights.append(pheromone_factor * heuristic_factor) # prefer going to
        d_weights = np.array(d_weights)
        d_weights = d_weights / d_weights.sum()
        return self.rand.choices(directions, weights = d_weights)[0]


    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)

        while self.current_position != self.end and self.steps < self.step_limit:
            direction = self.pick_direction()
            route = self.make_move(direction, route)
            self.visited_counts[self.current_position]  = self.visited_counts.get(self.current_position, 0) + 1

        terminated = self.current_position == self.end
        return route, terminated


In [89]:
class IntelligentAntColonyOptimization:

    """
    Constructs a new optimization object using the ant algorithm
    @param maze: the maze (environment) for ants
    @param ants_per_gen: the number of ants per generation (between update of pheromones)
    @param generations: the total number of generations of ants (pheromone updates)
    @param q: the normalization factor for the amount of dropped pheromone
    @param evaporation: the evaporation factor for the pheromones
    """
    def __init__(self, maze, ants_per_gen, generations, q, evaporation, step_limit, patience, threshold):
        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation
        self.step_limit = step_limit
        self.patience = patience
        self.threshold = threshold
        self.waited = 0

    """
    Loop that starts the shortest path process
    @param path_specification: description of the route we wish to optimize
    @return the optimized route according to the ACO algorithm
    """
    def find_shortest_route(self, path_specification, pheromone_exp, heuristic_exp):
        self.maze.reset()

        best = None
        prev_best = float('inf')
        best_size = float('inf')

        all_routes = []

        for generation in range(self.generations):

            print("-----------Generation " + str(generation) + "------------")

            curr_routes = []

            for ant_nr in range(self.ants_per_gen):
                ant = IntelligentAnt(self.maze, path_specification, self.step_limit, pheromone_exp, heuristic_exp)

                route, terminated = ant.find_route()

                print("Ant " + str(ant_nr))

                if terminated:
                    curr_routes.append(route)
                    all_routes.append(route)
                    if route.size() < best_size:
                        best = route
                        best_size = route.size()
                        print("New best route found of length " + str(route.size()))
                    else:
                        print("Worse route found of length " + str(route.size()))
                else:
                    print("Did not exit")

            increase = prev_best - best_size
            if increase > self.threshold:
                self.waited = 0
            else:
                self.waited += 1
                if self.waited == self.patience:
                    print("Stopped early due to no increase")
                    return best, len(all_routes)

            prev_best = best_size

            self.maze.evaporate(self.evaporation)
            self.maze.add_pheromone_routes(curr_routes, self.q)
            # if curr_routes:
            #     best_route_per_gen = min(curr_routes, key=lambda r: r.size())
            #     self.maze.add_pheromone_route(best_route_per_gen, self.q)

        return best, len(all_routes)

In [90]:
def verify_route(route, end):
    pos = route.get_start()
    dir_list = route.get_route()
    for direction in dir_list:
        pos = pos.add_direction(direction)
    return pos == end

In [91]:
# Please keep your parameters for the ACO easily changeable here
gen = 1 # ants per generation
no_gen = 40 # nr of generations
q = 500
evap = 0.2
step_scale_factor = 200
pheromone_exp = 1
heuristic_exp = 4

# Construct the optimization objects
maze = Maze.create_maze("./../data/hard_maze.txt")
spec = PathSpecification.read_coordinates("./../data/hard_coordinates.txt")

max_steps = maze.width * maze.length * step_scale_factor

aco = IntelligentAntColonyOptimization(maze, gen, no_gen, q, evap, max_steps, 40, 3)

start_time = int(round(time.time() * 1000))
shortest_route, routes_found = aco.find_shortest_route(spec, pheromone_exp, heuristic_exp)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
print("Route size: " + str(shortest_route.size()))
print("Number of ants that got out: " + str(routes_found))

is_valid = verify_route(shortest_route, spec.get_end())

print("Route is valid: " + str(is_valid))

shortest_route.write_to_file("./../data/hard_solution.txt")

Ready reading maze file ./../data/hard_maze.txt
-----------Generation 0------------
Ant 0
New best route found of length 885
-----------Generation 1------------
Ant 0
Worse route found of length 919
-----------Generation 2------------
Ant 0
Worse route found of length 891
-----------Generation 3------------
Ant 0
Worse route found of length 1025
-----------Generation 4------------
Ant 0
Worse route found of length 965
-----------Generation 5------------
Ant 0
Worse route found of length 981
-----------Generation 6------------


KeyboardInterrupt: 

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

### 2.5 Parameter Optimization

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 17

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.6 The Final Route

#### Question 18

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.7 Synthesis

#### Question 19

In [0]:
# Please keep your parameters for the synthesis part easily changeable here
gen = 1
no_gen = 1
q = 1000
evap = 0.1

persist_file = "./../tmp/my_tsp"
tsp_path = "./../data/tsp_products.txt"
coordinates = "./../data/hard_coordinates.txt"

# Construct optimization
maze = Maze.create_maze("./../data/hard_maze.txt")
tsp_data = TSPData.read_specification(coordinates, tsp_path)
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

# Run optimization and write to file
tsp_data.calculate_routes(aco)
tsp_data.write_to_file(persist_file)

# Read from file and print
tsp_data2 = TSPData.read_from_file(persist_file)
print(tsp_data == tsp_data2)

# Solve TSP using your own paths file
ga = GeneticAlgorithm(generations, population_size)
solution = ga.solve_tsp(tsp_data2)
tsp_data2.write_action_file(solution, "./../data/tsp_solution.txt")

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

## Part 3: Open Questions
### 3.1 Reflection

#### Question 20

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 21

<div style="background-color:#f1be3e">

_Write your answer here._

### 3.2 Pen and Paper

#### Question 22

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

### 3.3 Division of Work

#### Question 23

<div style="background-color:#f1be3e">


|          Component          |  Name A   |  Name B   |  Name C   |  Name D   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     A     |     B     |     C     |     D     |
| Code (implementation)       |     A     |     B     |     C     |     D     |
| Code (validation)           |     A     |     B     |     C     |     D     |
| Experiments (execution)     |     A     |     B     |     C     |     D     |
| Experiments (analysis)      |     A     |     B     |     C     |     D     |
| Experiments (visualization) |     A     |     B     |     C     |     D     |
| Report (original draft)     |     A     |     B     |     C     |     D     |
| Report (reviewing, editing) |     A     |     B     |     C     |     D     |

### References

<div style="background-color:#f1be3e">

**If you made use of any non-course resources, cite them below.**